### Extract

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.webdriver import WebDriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
from loguru import logger
from time import sleep

class CustomOptions:
    """
    Classe responsável por aplicar as configurações do navegador Chrome para o WebDriver.

    Atributos:
    ----------
    chrome_options : Options
        Instância de opções do Chrome configurada com diversos argumentos.
    """

    def __init__(self) -> None:
        self.chrome_options = Options()
        self.chrome_options.add_argument("--disable-gpu")
        self.chrome_options.add_argument("--disable-3d-apis")
        self.chrome_options.add_argument("--allow-insecure-localhost")
        self.chrome_options.add_argument("--log-level=3")
        self.chrome_options.add_argument(
            "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.6167.140 Safari/537.36"
        )
        self.chrome_options.add_argument("--headless")

    def espera(self, driver: WebDriver) -> WebDriverWait:
        """
        Configura o tempo máximo de espera para que elementos estejam presentes na url especificada antes de interagir com eles.

        Parâmetros:
        -----------
        driver : WebDriver
            Instância do WebDriver que será usada para configurar a espera.

        Retorna:
        --------
        WebDriverWait
            Instância configurada de WebDriverWait.
        """
        self.tempo = 10
        return WebDriverWait(driver, self.tempo)
    
class DriverManager:
    """
    Classe responsável por inicializar e fornecer acesso ao WebDriver e suas configurações.

    Atributos:
    ----------
    __driver : WebDriver
        Instância do navegador configurada com as opções fornecidas.
    __options : CustomOptions
        Instância de opções personalizadas para o navegador.
    """
    def __init__(self, options:CustomOptions) -> None:
        self.__driver = webdriver.Chrome(options=options.chrome_options)
        self.__options = options

    def get_driver(self) -> None:
        """
        Retorna a instância do WebDriver.

        Retorna:
        --------
        WebDriver
        """
        return self.__driver
    
    def get_options(self) -> None:
        """
        Retorna a instância de opções do navegador.

        Retorna:
        --------
        CustomOptions
        """
        return self.__options

class ScraperLinks:
    """
    Classe responsável por gerenciar o navegador e realizar o processo de scraping.

    Depende de instâncias configuradas de WebDriver e CustomOptions, fornecidas por DriverManager.

    Atributos:
    ----------
    driver : WebDriver
        Instância do WebDriver configurada com opções personalizadas.
    options : CustomOptions
        Instância da classe CustomOptions contendo as opções de configuração do navegador.
    pagina : int
        Número da página atual que está sendo processada.
    base_url : str
        URL base do site a ser raspado.
    dados_coletados : dict
        Dicionário que armazena os dados coletados como links dos produtos.
    """
    
    def __init__(self, driver: DriverManager) -> None:
        self.__driver = driver.get_driver()
        self.__options = driver.get_options()
        self.__pagina = 1
        self.base_url = 'https://casa.sapo.pt/comprar-apartamentos/porto/'
        self.__dados_coletados = {"link": []}

    def __acessar_url(self, url=None) -> None:
        """
        Acessa a URL especificada. Se nenhuma URL for passada, acessa a URL padrão configurada com base na página atual.

        Parâmetros:
        -----------
        url : str, opcional
            URL que será acessada. Se não especificada, será gerada a URL padrão baseada na página atual.

        Retorna:
        --------
        None
        """

        if url is None:
            url = f"{self.base_url}?pn={self.__pagina}"
        self.__driver.get(url)

    def __coletar_links(self) -> None:
        """
        Coleta todos os cartões de produtos presentes na página e extrai suas informações.

        Retorna:
        --------
        None
        """
        espera = self.__options.espera(self.__driver)
        aptos = espera.until(
            EC.presence_of_all_elements_located((
                By.XPATH,
                "//div[contains(@class, 'property-info-content')]"
            ))
        )

        for ap in aptos:
            if ap.is_displayed():
                link_element = ap.find_element(By.XPATH, ".//a[contains(@class, 'property-info')]")
                href = link_element.get_attribute("href")
                self.__dados_coletados['link'].append(href)

    def __valida_ultima_pagina(self) -> bool:
        """
        Verifica se a última página de resultados foi alcançada.

        Retorna:
        --------
        bool
            True se a última página foi alcançada, caso contrário False.
        """
        espera = self.__options.espera(self.__driver)
        try:
            ultima_pagina = espera.until(
                EC.presence_of_element_located((
                    By.XPATH,
                    "//span[contains(@class, 'disabled') and contains(., 'Seguinte')]"
                ))
            )

            if ultima_pagina:
                logger.info(
                    f"Ultima página: {self.__pagina}.\nDados extraídos com sucesso."
                )
                return True
        except:
            pass

    def __proxima_pagina(self) -> None:
        """
        Avança para a próxima página de resultados e acessa a URL correspondente.

        Retorna:
        --------
        None
        """
        self.__pagina += 1
        proxima_pagina = f"{self.base_url}?pn={self.__pagina}"
        self.__acessar_url(proxima_pagina)

    def scraping_links(self) -> None:
        """
        Executa o processo completo de scraping, passando por todas as páginas disponíveis até a última.

        Retorna:
        --------
        None
        """
        self.__acessar_url()
        while True:
            self.__coletar_links()

            logger.info(f"Coleta da página {self.__pagina} realizada com sucesso..")
            sleep(2)

            if self.__valida_ultima_pagina():
                break
            else:
                self.__proxima_pagina()

    def get_links(self) -> list[str]:
        """
        Retorna a lista de links coletados.

        Retorna:
        --------
        list[str]
        """

        return self.__dados_coletados["link"]


class ManipuladorArquivos:

    def get_links_from_csv(self, diretorio: str, arquivo: str) -> pd.DataFrame:
        
        links = pd.read_csv(f'{diretorio}/{arquivo}', sep=';')

        return links

class ScrapperInfo:

    def __init__(self, driver: DriverManager) -> None:
        self.__driver = driver.get_driver()
        self.__options = driver.get_options()
        self.links = ManipuladorArquivos().get_links_from_csv(diretorio='data', arquivo='links-aptos.csv') # Ajustar isso. Preciso chamar o método "percorre_links()" com o nome do arquivo como argumento.
        self.__dados_coletados = {"descricao": [], "dados_imovel": [], "caracteristicas": [], "coordenadas": [], "link": [], "preco": [], "anunciante": []}

    def __coleta_descricao(self) -> None:
        try:
            espera = self.__options.espera(self.__driver)
            elem = espera.until(
                EC.presence_of_element_located((
                    By.XPATH,
                    "//div[contains(@class, 'detail-section') and contains(@class, 'detail-title')]"
                ))
            )
            descricao = elem.find_element(By.TAG_NAME, "h1").text
        except Exception:
            descricao = None

        self.__dados_coletados['descricao'].append(descricao)

    def __coleta_preco(self) -> None:
        try:
            espera = self.__options.espera(self.__driver)
            elem = espera.until(
                EC.presence_of_element_located((
                    By.XPATH,
                    "//div[contains(@class, 'detail-section') and contains(@class, 'detail-title')]"
                ))
            )
            preco = elem.find_element(By.XPATH, "//div[contains(@class, 'detail-title-price-value')]").text
        except Exception:
            preco = None

        self.__dados_coletados['preco'].append(preco)

    def __coleta_dados_imovel(self) -> None:
        try:
            espera = self.__options.espera(self.__driver)
            main_elem = espera.until(
                EC.presence_of_element_located((
                    By.XPATH,
                    "//div[contains(@class, 'detail-main-features-list')]"
                ))
            )
            elementos = main_elem.find_elements(By.XPATH, ".//div[contains(@class, 'detail-main-features-item')]")

            dados = {}
            for elem in elementos:
                titulo_elem = elem.find_elements(By.XPATH, ".//div[contains(@class, 'detail-main-features-item-title')]")
                valor_elem = elem.find_elements(By.XPATH, ".//div[contains(@class, 'detail-main-features-item-value')]")
                item = titulo_elem[0].text.strip() if titulo_elem else None
                valor = valor_elem[0].text.strip() if valor_elem else None
                if item and valor:
                    dados[item] = valor
        except Exception:
            dados = None

        self.__dados_coletados['dados_imovel'].append(dados)

    def __percorre_e_coleta_caracteristicas(self) -> None:
        try:
            espera = self.__options.espera(self.__driver)
            menu = espera.until(EC.presence_of_element_located((By.XPATH, ".//div[contains(@class, 'detail-features-menu-content')]")))
            abas = menu.find_elements(By.TAG_NAME, "span")

            dados_coletados = {}
            for aba in abas:
                aba_name = self.__trata_nome_aba(aba.text)
                self.__driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", aba)
                sleep(0.5)
                self.__driver.execute_script("arguments[0].click();", aba)
                coletados = self.__coleta_caracteristicas()
                dados_coletados[aba_name] = coletados
        except Exception:
            dados_coletados = None

        self.__dados_coletados['caracteristicas'].append(dados_coletados)

    def __trata_nome_aba(self, nome: str) -> str:
        nome_tratado = nome.lower().replace(' ', '-').translate(str.maketrans('áãâàéêíóõôúç', 'aaaeeiiooouc'))
        return nome_tratado

    def __coleta_caracteristicas(self) -> None:
        espera = self.__options.espera(self.__driver)

        main_elem = espera.until(
            EC.presence_of_element_located((
                By.XPATH,
                "//div[contains(@class, 'detail-section') and contains(@class, 'detail-features')]"
            ))
        )

        menu_items = main_elem.find_element(By.XPATH, ".//div[contains(@class, 'detail-features-items')]")
        items = menu_items.find_elements(By.XPATH, ".//div[contains(@class, 'detail-features-item')]")

        items_coletados = []

        for item in items:
            items_coletados.append(item.text)

        return items_coletados

    def __coleta_coords(self) -> None:
        try:
            espera = self.__options.espera(self.__driver)
            map = espera.until(
            EC.presence_of_element_located((
                By.XPATH,
                "//div[contains(@id, 'objMap')]"
                ))
            )

            lat = map.get_attribute('data-latitude')
            long = map.get_attribute('data-longitude')

            coords = {'lat': lat, 'long': long}
        except Exception:
            coords = None

        self.__dados_coletados['coordenadas'].append(coords)

    def __coleta_link(self, link: str) -> None:
        self.__dados_coletados['link'].append(link)

    def __coleta_anunciante(self) -> None:
        try:
            espera = self.__options.espera(self.__driver)
            elem = espera.until(
            EC.presence_of_element_located((
                By.XPATH,
                "//div[contains(@class, 'detail-owner-name')]"
                ))
            )

            anunciante = elem.text.strip()
        except Exception:  
            anunciante = None

        self.__dados_coletados['anunciante'].append(anunciante)

    def __reseta_driver(self) -> None:
        logger.info("Recriando driver para evitar crash de aba.")
        self.__driver.quit()
        self.__driver = webdriver.Chrome(options=self.__options.chrome_options)

    def percorre_links(self, index_inicial=None, index_final=None) -> None: # após testes, encapsular método
        for i, link in enumerate(self.links['links'][index_inicial:index_final]):
            if i > 0 and i % 500 == 0:               
                self.__reseta_driver()

            try:
                self.__driver.get(link)
                self.__coleta_descricao()
                self.__coleta_preco()
                self.__coleta_dados_imovel()
                self.__coleta_coords()
                self.__percorre_e_coleta_caracteristicas()
                self.__coleta_link(link)
                self.__coleta_anunciante()
                logger.info(f'Coleta do link "{i+1}" realizada com sucesso.')
            except Exception as e:
                logger.error(f'Erro ao processar o link {i+1} - {e}', exc_info=True)


    def get_dados(self) -> dict[str, list]:
        """
        Retorna os dados coletados durante o processo de Scrapping

        Retorna:
        --------
        list[str]
        """

        return self.__dados_coletados



## Load

In [14]:
import pandas as pd
import os

class FormatoArmazenamento:
    """
    Classe base para diferentes formatos de armazenamento de dados.
    Contém o método para armazenar arquivos no formato CSV.
    """

    def armazenar_csv(self, df: pd.DataFrame, diretorio: str, nome_arquivo: str) -> str:
        """
        Armazena um DataFrame como um arquivo CSV no diretório e nome de arquivo especificados.

        Parâmetros:
        -----------
        df : pd.DataFrame
            DataFrame que será salvo.
        diretorio : str
            Caminho do diretório onde o arquivo será salvo.
        nome_arquivo : str
            Nome do arquivo CSV a ser criado.

        Retorna:
        --------
        str
            Representação do CSV como string.
        """
        dados_transformados = df.to_csv(
            os.path.join(diretorio, nome_arquivo),
            sep=";",
            encoding="utf-8",
            header=True,
            index=False,
        )

        return dados_transformados


class Armazenamento(FormatoArmazenamento):
    """
    Classe responsável por gerenciar o armazenamento de dados em disco.

    Atributos:
    ----------
    diretorio : str
        Caminho do diretório onde os dados serão salvos.
    """

    def __init__(self, diretorio: str) -> None:
        """
        Inicializa a instância com o diretório de destino.

        Parâmetros:
        -----------
        diretorio : str
            Caminho do diretório onde os arquivos serão armazenados.
        """
        self.diretorio = diretorio

    def checa_diretorio(self) -> None:
        """
        Verifica se o diretório existe. Se não existir, cria.
        """
        os.makedirs(self.diretorio, exist_ok=True)

    def gerar_csv_links(self, dados: list[str]) -> None:
        """
        Armazena os dados coletados durante o processo de scraping em um arquivo CSV.

        Parâmetros:
        -----------
        dados : list[str]
            Lista de links coletados a serem armazenados.
        """
        df = pd.DataFrame(dados, columns=['links'])
        df = df.drop_duplicates(subset=["links"])

        self.checa_diretorio()
        self.armazenar_csv(df=df, diretorio=self.diretorio, nome_arquivo="links-aptos.csv")



    def gerar_csv_dados(self, dados: list[any]) -> None:
        
        df = pd.DataFrame(dados) #columns=['descricao', 'dados_imovel', 'caracteristicas', 'coordenadas', 'link', 'preco'] #Ajustar isso aqui para coletar as colunas automaticamente. Sempre que adiciona uma variável, precisa incluir manualmente
        df = df.drop_duplicates(subset=["link"])
        
        self.checa_diretorio()
        self.armazenar_csv(df=df, diretorio=self.diretorio, nome_arquivo="dados-aptos.csv")

In [25]:
a = dados.links['links'][None:None]

print(a)

0       https://gespub.casa.sapo.pt/v3/webinterface/cl...
1       https://gespub.casa.sapo.pt/v3/webinterface/cl...
2       https://gespub.casa.sapo.pt/v3/webinterface/cl...
3       https://gespub.casa.sapo.pt/v3/webinterface/cl...
4       https://gespub.casa.sapo.pt/v3/webinterface/cl...
                              ...                        
4751    https://casa.sapo.pt/comprar-apartamento-t1-po...
4752    https://casa.sapo.pt/comprar-apartamento-t1-po...
4753    https://casa.sapo.pt/comprar-apartamento-t1-po...
4754    https://casa.sapo.pt/comprar-apartamento-t2-po...
4755    https://casa.sapo.pt/comprar-apartamento-t3-po...
Name: links, Length: 4756, dtype: object


In [15]:
carregar = Armazenamento(diretorio='data')
options = CustomOptions()
driver = DriverManager(options)

links = ScraperLinks(driver)
dados = ScrapperInfo(driver)

In [5]:
dados.percorre_links()

2025-12-11 06:43:46.973 | INFO     | __main__:percorre_links:370 - Coleta do link "1" realizada com sucesso.
2025-12-11 06:43:51.626 | INFO     | __main__:percorre_links:370 - Coleta do link "2" realizada com sucesso.
2025-12-11 06:44:01.614 | INFO     | __main__:percorre_links:370 - Coleta do link "3" realizada com sucesso.
2025-12-11 06:44:10.126 | INFO     | __main__:percorre_links:370 - Coleta do link "4" realizada com sucesso.
2025-12-11 06:44:18.428 | INFO     | __main__:percorre_links:370 - Coleta do link "5" realizada com sucesso.
2025-12-11 06:44:25.173 | INFO     | __main__:percorre_links:370 - Coleta do link "6" realizada com sucesso.
2025-12-11 06:44:31.646 | INFO     | __main__:percorre_links:370 - Coleta do link "7" realizada com sucesso.
2025-12-11 06:44:38.878 | INFO     | __main__:percorre_links:370 - Coleta do link "8" realizada com sucesso.
2025-12-11 06:44:47.059 | INFO     | __main__:percorre_links:370 - Coleta do link "9" realizada com sucesso.
2025-12-11 06:44:53

In [9]:
dados_proc = dados.get_dados()

for chave, lista in dados_proc.items():
    print(chave, len(lista))

descricao 4753
dados_imovel 4753
caracteristicas 4744
coordenadas 4744
link 4744
preco 4753


In [7]:
carregar.gerar_csv_dados(dados.get_dados())

ValueError: All arrays must be of the same length

In [4]:
df = pd.read_csv("data/dados-aptos.csv", sep=";")
print(df["dados_imovel"].iloc[0])

{'ESTADO': 'Usado', 'ÁREA ÚTIL': '127m²', 'ÁREA BRUTA': '156m²', 'REFERÊNCIA': 'CasaSAPO_APA_938', 'CERTIFICAÇÃO ENERGÉTICA': 'C', 'VISUALIZAÇÕES': '148', 'CLIQUES': '8', 'PUBLICADO EM': '14/11/2025'}
